In [1]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch, export_sdf

In [24]:
df = pd.read_excel('./failed_molecules.xlsx', sheet_name=0)
df

,Unnamed: 0,compound_id,smiles,status,mop_file,out_file,arc_file,error_message,return_code,duration_s
0,25,ID0026,ClC(Cl)C(c1ccc(cc1)Cl)c1ccc(cc1)Cl,success,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,0,103.375
1,284,ID0285,CCN(CC)C(=O)C(C)Oc1cccc2ccccc12,timeout,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,MOPAC timed out after 900s.,-1,900.019
2,595,ID0596,Oc1cccc2c1C(=O)c1c(cccc1C2=O)O,success,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,0,53.509
3,1245,ID1246,CC1COC(Cn2cncn2)(O1)c1ccc(cc1Cl)Oc1ccc(cc1)Cl,success,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,0,182.671
4,1321,ID1322,CC1OC(OC2CC3OC(O)(CC(O)CC(O)C(O)CCC(O)CC(O)CC(...,timeout,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,MOPAC timed out after 900s.,-1,900.026
5,1325,ID1326,CCC1NC(=O)C(C(O)C(C)CC=CC)N(C)C(=O)C(C(C)C)N(C...,timeout,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,MOPAC timed out after 900s.,-1,900.029
6,1339,ID1340,CCC(C)C1OC2(CC3CC(CC=C(C)C(OC4CC(OC)C(OC5CC(OC...,timeout,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,MOPAC timed out after 900s.,-1,900.023
7,1341,ID1342,CC1CC2(OC(O)C3(C)OC23)OC2CC3(C)C4CCC5C(C)(C)C(...,timeout,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,MOPAC timed out after 900s.,-1,900.019
8,1371,ID1372,CCC(C)C1OC2(CC3CC(C\C=C(/C)\C(OC4CC(OC)C(OC5CC...,timeout,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,MOPAC timed out after 900s.,-1,900.014
9,1373,ID1374,COC1CC(CCC1O)CC(C)C1CC(=O)C(C)C=C(C)C(O)C(OC)C...,timeout,C:\Users\macie\Documents\GitHub\smi_to_mop\pxr...,NaN,NaN,MOPAC timed out after 900s.,-1,905.372


In [25]:
work_dir = 'pxr_dataset_failed_jobs'

In [ ]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df,
    smiles_col="smiles",
    id_col="compound_id",
    keywords="PM7 PRECISE GEO-OK",
    workdir=work_dir,
    auto_charge=True,   # ← detects [NH4+], [O-], etc. automatically
    overwrite=True,
    max_workers=6
)

In [29]:
# verify if anything failed
print(prep[prep['status'] != 'ready'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, error_message, duration_s]
Index: []


In [30]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir=work_dir,
    overwrite=True,
    timeout=2000,
    max_workers=8,
)

[ID1340] MOPAC timed out.
[ID1322] MOPAC timed out.
[ID1374] MOPAC timed out.
[ID1533] MOPAC timed out.
[ID1573] MOPAC timed out.
[ID1651] MOPAC timed out.


In [32]:
results[results['arc_file'].isna()].to_excel('twice_failed_molecules.xlsx')
results[results['arc_file'].isna()]['compound_id']

2     ID0596
4     ID1322
5     ID1326
6     ID1340
8     ID1372
9     ID1374
10    ID1533
11    ID1573
13    ID1651
18    ID2213
Name: compound_id, dtype: str

In [17]:
# Step 3: export optimised geometries to SDF
sdf = export_sdf(
    workdir="mopac_jobs",
    overwrite=True,
    max_workers=6,
)

In [18]:
print(sdf[sdf['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, arc_file, sdf_file, status, error_message, duration_s]
Index: []


In [19]:
from pathlib import Path
import shutil

source_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\mopac_jobs")
target_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\sdf_all")

target_dir.mkdir(exist_ok=True)

for sdf_file in source_dir.rglob("*.sdf"):
    shutil.copy2(sdf_file, target_dir / sdf_file.name)